<a href="https://colab.research.google.com/github/Tanuja953/Operating-System-Assignment/blob/main/ooperating_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Operating Systems Assignment





## Practical 1: Process Creation using fork() and exec()

**Aim:** To study and demonstrate process creation and program execution using the fork() and exec() system calls.

In [ ]:
%%writefile fork_exec.py
import os
import sys
import warnings
# Suppress the fork warning in Google Colab
warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
    message="This process.*fork.*"
)

print(f"Parent process: PID = {os.getpid()}", flush=True)

try:
    pid = os.fork()
except OSError:
    # fork failed
    print("fork failed")
    sys.exit(1)

if pid == 0:
    # Child process
    print(f"Child process: PID = {os.getpid()}, Parent PID = {os.getppid()}", flush=True)
    print("Child is about to exec 'ls -l'", flush=True)
    os.execlp("ls", "ls", "-l")
    # execlp only returns on error
    print("exec failed")
    os._exit(1)
else:
    # Parent process
    print(f"Parent waiting for child (PID = {pid}) to finish...", flush=True)
    _, status = os.wait()
    if os.WIFEXITED(status):
        print(f"Child exited with status {os.WEXITSTATUS(status)}")
    print("Parent process resumes and exits.")

Writing fork_exec.py


In [ ]:
!python3 fork_exec.py

Parent process: PID = 4249
Parent waiting for child (PID = 4250) to finish...
Child process: PID = 4250, Parent PID = 4249
Child is about to exec 'ls -l'
total 8
-rw-r--r-- 1 root root  913 Nov 24 18:15 fork_exec.py
drwxr-xr-x 1 root root 4096 Nov 18 15:58 sample_data
Child exited with status 0
Parent process resumes and exits.


## Practical 2: Thread Synchronization using a Lock

**Aim:** To study thread synchronization and demonstrate the use of a lock to protect shared resources.

In [ ]:
import threading

NUM_THREADS = 4

print_lock = threading.Lock()   # keeps output lines from different threads separate


# argument and result passed to each thread
class PartialSum:
    def __init__(self, start, end):
        self.start = start
        self.end = end
        self.sum = 0


def sum_array(p):
    p.sum = 0
    for i in range(p.start, p.end):
        p.sum += i
    with print_lock:
        print(f"{threading.current_thread().name}: sum[{p.start}..{p.end}) = {p.sum}")


def main():
    n = 1000000
    chunk = n // NUM_THREADS
    total = 0
    threads = []
    data = []

    for i in range(NUM_THREADS):
        start = i * chunk
        end = n if i == NUM_THREADS - 1 else (i + 1) * chunk
        data.append(PartialSum(start, end))
        t = threading.Thread(target=sum_array, args=(data[i],), name=f"Thread {i + 1}")
        threads.append(t)
        t.start()

    for i in range(NUM_THREADS):
        threads[i].join()
        total += data[i].sum

    print(f"Total sum (0 to {n - 1}) computed using {NUM_THREADS} threads = {total}")


if __name__ == "__main__":
    main()

Thread 2: sum[250000..500000) = 93749875000
Thread 1: sum[0..250000) = 31249875000
Thread 3: sum[500000..750000) = 156249875000
Thread 4: sum[750000..1000000) = 218749875000
Total sum (0 to 999999) computed using 4 threads = 499999500000


## Practical 3: CPU Scheduling Algorithms: FCFS, SJF, Priority and Round Robin

**Aim:** To implement and compare CPU scheduling algorithms: FCFS, SJF, Priority Scheduling and Round Robin.

In [ ]:
class Process:
    def __init__(self, pid, at, bt, pr):
        self.pid, self.at, self.bt, self.pr = pid, at, bt, pr
        self.ct = self.tat = self.wt = self.rem = 0


def print_result(p):
    n = len(p)
    total_wt = total_tat = 0
    print()
    print(f"{'PID':<5}{'AT':<4}{'BT':<4}{'CT':<4}{'TAT':<5}{'WT'}")
    for i in range(n):
        print(f"{p[i].pid:<5}{p[i].at:<4}{p[i].bt:<4}{p[i].ct:<4}{p[i].tat:<5}{p[i].wt}")
        total_wt += p[i].wt
        total_tat += p[i].tat
    print(f"Average Waiting Time    = {total_wt / n:.2f}")
    print(f"Average Turnaround Time = {total_tat / n:.2f}")


def fcfs(p):
    time = 0
    for i in range(len(p)):
        if time < p[i].at:
            time = p[i].at
        time += p[i].bt
        p[i].ct = time
        p[i].tat = p[i].ct - p[i].at
        p[i].wt = p[i].tat - p[i].bt
    print_result(p)


def sjf(p):
    n = len(p)
    completed, time = 0, 0
    done = [False] * n
    while completed < n:
        idx, min_bt = -1, 10**9
        for i in range(n):
            if not done[i] and p[i].at <= time and p[i].bt < min_bt:
                min_bt = p[i].bt
                idx = i
        if idx == -1:
            time += 1
            continue
        time += p[idx].bt
        p[idx].ct = time
        p[idx].tat = p[idx].ct - p[idx].at
        p[idx].wt = p[idx].tat - p[idx].bt
        done[idx] = True
        completed += 1
    print_result(p)


def priority_scheduling(p):
    n = len(p)
    completed, time = 0, 0
    done = [False] * n
    while completed < n:
        idx, best = -1, 10**9
        for i in range(n):
            if not done[i] and p[i].at <= time and p[i].pr < best:
                best = p[i].pr
                idx = i
        if idx == -1:
            time += 1
            continue
        time += p[idx].bt
        p[idx].ct = time
        p[idx].tat = p[idx].ct - p[idx].at
        p[idx].wt = p[idx].tat - p[idx].bt
        done[idx] = True
        completed += 1
    print_result(p)


def round_robin(p, quantum):
    n = len(p)
    time, completed = 0, 0
    for proc in p:
        proc.rem = proc.bt

    queue = []
    in_queue = [False] * n

    # start with process(es) arriving at time 0, in PID order
    for i in range(n):
        if p[i].at <= time:
            queue.append(i)
            in_queue[i] = True

    while completed < n:
        if not queue:
            time += 1
            for i in range(n):
                if not in_queue[i] and p[i].rem > 0 and p[i].at <= time:
                    queue.append(i)
                    in_queue[i] = True
            continue
        idx = queue.pop(0)
        slice_ = min(p[idx].rem, quantum)
        time += slice_
        p[idx].rem -= slice_

        for i in range(n):
            if not in_queue[i] and p[i].rem > 0 and p[i].at <= time and i != idx:
                queue.append(i)
                in_queue[i] = True

        if p[idx].rem > 0:
            queue.append(idx)
        else:
            p[idx].ct = time
            p[idx].tat = p[idx].ct - p[idx].at
            p[idx].wt = p[idx].tat - p[idx].bt
            completed += 1
            in_queue[idx] = False
    print_result(p)


def main():
    n = int(input("Enter number of processes: "))
    p = []
    for i in range(n):
        # Replaced commas with spaces to handle formats like '0,5,2' or '0 5 2' seamlessly
        raw_input = input(f"Process {i + 1} - Arrival Time, Burst Time, Priority: ").replace(",", " ")
        at, bt, pr = map(int, raw_input.split())
        p.append(Process(i + 1, at, bt, pr))

    print("\nMenu:\n1. FCFS\n2. SJF (Non-preemptive)")
    print("3. Priority (Non-preemptive)\n4. Round Robin")
    choice = int(input("Enter choice: "))

    if choice == 1:
        fcfs(p)
    elif choice == 2:
        sjf(p)
    elif choice == 3:
        priority_scheduling(p)
    elif choice == 4:
        quantum = int(input("Enter time quantum: "))
        round_robin(p, quantum)
    else:
        print("Invalid choice")


if __name__ == "__main__":
    main()

Enter number of processes: 4
Process 1 - Arrival Time, Burst Time, Priority: 0 5 2
Process 2 - Arrival Time, Burst Time, Priority: 1 3 1
Process 3 - Arrival Time, Burst Time, Priority: 2 8 3
Process 4 - Arrival Time, Burst Time, Priority: 3 6 4

Menu:
1. FCFS
2. SJF (Non-preemptive)
3. Priority (Non-preemptive)
4. Round Robin
Enter choice: 4
Enter time quantum: 4

PID  AT  BT  CT  TAT  WT
1    0   5   16  16   11
2    1   3   7   6    3
3    2   8   20  18   10
4    3   6   22  19   13
Average Waiting Time    = 9.25
Average Turnaround Time = 14.75


## Practical 4: Producer–Consumer Problem using Semaphores

**Aim:** To understand and implement the Producer–Consumer problem using semaphores for synchronization.

In [ ]:
import threading
import time

BUFFER_SIZE = 5
ITEMS = 10

buffer = [0] * BUFFER_SIZE
in_idx = 0
out_idx = 0

empty = threading.Semaphore(BUFFER_SIZE)
full = threading.Semaphore(0)
mutex = threading.Semaphore(1)


def producer():
    global in_idx
    for i in range(1, ITEMS + 1):
        empty.acquire()
        mutex.acquire()

        buffer[in_idx] = i
        print(f"Producer produced item {i} at slot {in_idx}")
        in_idx = (in_idx + 1) % BUFFER_SIZE

        mutex.release()
        full.release()
        time.sleep(0.1)


def consumer():
    global out_idx
    for i in range(1, ITEMS + 1):
        full.acquire()
        mutex.acquire()

        item = buffer[out_idx]
        print(f"\t\tConsumer consumed item {item} from slot {out_idx}")
        out_idx = (out_idx + 1) % BUFFER_SIZE

        mutex.release()
        empty.release()
        time.sleep(0.15)


def main():
    prod = threading.Thread(target=producer)
    cons = threading.Thread(target=consumer)

    prod.start()
    cons.start()

    prod.join()
    cons.join()

    print("All items produced and consumed.")


if __name__ == "__main__":
    main()

Producer produced item 1 at slot 0
		Consumer consumed item 1 from slot 0
Producer produced item 2 at slot 1
		Consumer consumed item 2 from slot 1
Producer produced item 3 at slot 2
		Consumer consumed item 3 from slot 2
Producer produced item 4 at slot 3
Producer produced item 5 at slot 4
		Consumer consumed item 4 from slot 3
Producer produced item 6 at slot 0
		Consumer consumed item 5 from slot 4
Producer produced item 7 at slot 1
Producer produced item 8 at slot 2
		Consumer consumed item 6 from slot 0
Producer produced item 9 at slot 3
		Consumer consumed item 7 from slot 1
Producer produced item 10 at slot 4
		Consumer consumed item 8 from slot 2
		Consumer consumed item 9 from slot 3
		Consumer consumed item 10 from slot 4
All items produced and consumed.


## Practical 5: Readers–Writers Problem using Semaphores

**Aim:** To understand and implement the Readers–Writers problem using semaphores.

In [ ]:
import threading
import time

mutex = threading.Semaphore(1)
wrt = threading.Semaphore(1)
print_lock = threading.Lock()   # keeps output lines from different threads separate
read_count = 0
shared_data = 0


def reader(rid):
    global read_count

    mutex.acquire()
    read_count += 1
    if read_count == 1:
        wrt.acquire()          # first reader locks the resource
    mutex.release()

    with print_lock:
        print(f"Reader {rid}: read shared_data = {shared_data}")
    time.sleep(0.2)

    mutex.acquire()
    read_count -= 1
    if read_count == 0:
        wrt.release()          # last reader unlocks the resource
    mutex.release()


def writer(wid):
    global shared_data

    wrt.acquire()
    shared_data += 1
    with print_lock:
        print(f"Writer {wid}: wrote shared_data = {shared_data}")
    time.sleep(0.3)
    wrt.release()


def main():
    w = [threading.Thread(target=writer, args=(1,)),
         threading.Thread(target=writer, args=(2,))]
    r = [threading.Thread(target=reader, args=(i + 1,)) for i in range(5)]

    w[0].start()
    for t in r:
        t.start()
    w[1].start()

    for t in r:
        t.join()
    w[0].join()
    w[1].join()


if __name__ == "__main__":
    main()

Writer 1: wrote shared_data = 1
Reader 1: read shared_data = 1
Reader 2: read shared_data = 1
Reader 3: read shared_data = 1
Reader 4: read shared_data = 1
Reader 5: read shared_data = 1
Writer 2: wrote shared_data = 2


## Practical 6: Dining Philosophers Problem

**Aim:** To study and implement the Dining Philosophers problem and understand synchronization among competing processes.

In [ ]:
import threading
import time

N = 5

forks = [threading.Semaphore(1) for _ in range(N)]
room = threading.Semaphore(N - 1)   # allows at most N-1 philosophers to sit and pick up forks
print_lock = threading.Lock()       # keeps output lines from different threads separate


def philosopher(pid):
    left = pid
    right = (pid + 1) % N

    for cycle in range(2):
        with print_lock:
            print(f"Philosopher {pid} is thinking")
        time.sleep(0.1)

        room.acquire()             # enter the "room" - avoids deadlock
        forks[left].acquire()      # pick up left fork
        forks[right].acquire()     # pick up right fork

        with print_lock:
            print(f"Philosopher {pid} is eating (fork {left} and fork {right})")
        time.sleep(0.15)

        forks[right].release()
        forks[left].release()
        room.release()

        with print_lock:
            print(f"Philosopher {pid} finished eating and put down forks")


def main():
    philo = [threading.Thread(target=philosopher, args=(i,)) for i in range(N)]

    for t in philo:
        t.start()
    for t in philo:
        t.join()


if __name__ == "__main__":
    main()

Philosopher 0 is thinking
Philosopher 1 is thinking
Philosopher 2 is thinking
Philosopher 3 is thinking
Philosopher 4 is thinking
Philosopher 0 is eating (fork 0 and fork 1)
Philosopher 2 is eating (fork 2 and fork 3)
Philosopher 0 finished eating and put down forks
Philosopher 0 is thinking
Philosopher 2 finished eating and put down forks
Philosopher 2 is thinking
Philosopher 3 is eating (fork 3 and fork 4)
Philosopher 1 is eating (fork 1 and fork 2)
Philosopher 3 finished eating and put down forks
Philosopher 3 is thinking
Philosopher 1 finished eating and put down forks
Philosopher 1 is thinking
Philosopher 4 is eating (fork 4 and fork 0)
Philosopher 2 is eating (fork 2 and fork 3)
Philosopher 4 finished eating and put down forks
Philosopher 4 is thinking
Philosopher 2 finished eating and put down forks
Philosopher 1 is eating (fork 1 and fork 2)
Philosopher 3 is eating (fork 3 and fork 4)
Philosopher 1 finished eating and put down forks
Philosopher 0 is eating (fork 0 and fork 1)
P

## Practical 7: Banker's Algorithm for Deadlock Avoidance

**Aim:** To implement the Banker's Algorithm and check whether the system is in a safe state to avoid deadlock.

In [ ]:
P = 5   # number of processes
R = 3   # number of resource types

available = [0] * R
maximum = [[0] * R for _ in range(P)]
allocation = [[0] * R for _ in range(P)]
need = [[0] * R for _ in range(P)]


def calculate_need():
    for i in range(P):
        for j in range(R):
            need[i][j] = maximum[i][j] - allocation[i][j]


def is_safe(safe_seq):
    work = available[:]
    finish = [False] * P

    count = 0
    while count < P:
        found = False
        for i in range(P):
            if not finish[i] and all(need[i][j] <= work[j] for j in range(R)):
                for k in range(R):
                    work[k] += allocation[i][k]
                safe_seq.append(i)
                count += 1
                finish[i] = True
                found = True
        if not found:
            return False   # no process could proceed: unsafe
    return True


def request_resources(pid, request):
    for j in range(R):
        if request[j] > need[pid][j]:
            print("Error: process has exceeded its maximum claim.")
            return False
        if request[j] > available[j]:
            print("Request denied: not enough resources available.")
            return False

    # tentatively allocate
    for j in range(R):
        available[j] -= request[j]
        allocation[pid][j] += request[j]
        need[pid][j] -= request[j]

    if is_safe([]):
        print("Request can be granted safely.")
        return True
    else:
        # rollback
        for j in range(R):
            available[j] += request[j]
            allocation[pid][j] -= request[j]
            need[pid][j] += request[j]
        print("Request denied: would lead to an unsafe state.")
        return False


def main():
    available[:] = map(int, input(f"Enter available resources of each type ({R} types): ").split())

    print(f"Enter Maximum matrix ({P} processes x {R} resources):")
    for i in range(P):
        maximum[i] = list(map(int, input().split()))

    print("Enter Allocation matrix:")
    for i in range(P):
        allocation[i] = list(map(int, input().split()))

    calculate_need()

    safe_seq = []
    if is_safe(safe_seq):
        print("System is in a SAFE state.")
        print("Safe sequence:", " ".join(f"P{i}" for i in safe_seq))
    else:
        print("System is in an UNSAFE state.")

    pid = int(input("\nEnter process number requesting resources: "))
    request = list(map(int, input(f"Enter request vector ({R} values): ").split()))

    request_resources(pid, request)


if __name__ == "__main__":
    main()

Enter available resources of each type (3 types): 3 3 2
Enter Maximum matrix (5 processes x 3 resources):
7 5 3
3 2 2
9 0 2
2 2 2
4 3 3
Enter Allocation matrix:
0 1 0
2 0 0
3 0 2
2 1 1
0 0 2
System is in a SAFE state.
Safe sequence: P1 P3 P4 P0 P2

Enter process number requesting resources: 1
Enter request vector (3 values): 1 0 2
Request can be granted safely.


## Practical 8: Memory Allocation: First Fit, Best Fit and Worst Fit

**Aim:** To implement and compare the First Fit, Best Fit and Worst Fit memory allocation strategies.

In [ ]:
def print_allocation(title, psize, allocation):
    print(f"\n--- {title} ---")
    print(f"{'Process No.':<13}{'Process Size':<14}Block No.")
    for i in range(len(psize)):
        block = f"B{allocation[i] + 1}" if allocation[i] != -1 else "Not Allocated"
        print(f"{'P' + str(i + 1):<13}{psize[i]:<14}{block}")


def first_fit(bsize, psize):
    block = bsize[:]
    allocation = [-1] * len(psize)

    for i in range(len(psize)):
        for j in range(len(block)):
            if block[j] >= psize[i]:
                allocation[i] = j
                block[j] -= psize[i]
                break
    print_allocation("First Fit", psize, allocation)


def best_fit(bsize, psize):
    block = bsize[:]
    allocation = [-1] * len(psize)

    for i in range(len(psize)):
        best_idx = -1
        for j in range(len(block)):
            if block[j] >= psize[i]:
                if best_idx == -1 or block[j] < block[best_idx]:
                    best_idx = j
        if best_idx != -1:
            allocation[i] = best_idx
            block[best_idx] -= psize[i]
    print_allocation("Best Fit", psize, allocation)


def worst_fit(bsize, psize):
    block = bsize[:]
    allocation = [-1] * len(psize)

    for i in range(len(psize)):
        worst_idx = -1
        for j in range(len(block)):
            if block[j] >= psize[i]:
                if worst_idx == -1 or block[j] > block[worst_idx]:
                    worst_idx = j
        if worst_idx != -1:
            allocation[i] = worst_idx
            block[worst_idx] -= psize[i]
    print_allocation("Worst Fit", psize, allocation)


def main():
    bn = int(input("Enter number of memory blocks: "))
    bsize = list(map(int, input("Enter sizes of blocks: ").split()))[:bn]

    pn = int(input("Enter number of processes: "))
    psize = list(map(int, input("Enter sizes of processes: ").split()))[:pn]

    first_fit(bsize, psize)
    best_fit(bsize, psize)
    worst_fit(bsize, psize)


if __name__ == "__main__":
    main()

Enter number of memory blocks: 5
Enter sizes of blocks: 100 500 200 300 600
Enter number of processes: 4
Enter sizes of processes: 212 417 112 426

--- First Fit ---
Process No.  Process Size  Block No.
P1           212           B2
P2           417           B5
P3           112           B2
P4           426           Not Allocated

--- Best Fit ---
Process No.  Process Size  Block No.
P1           212           B4
P2           417           B2
P3           112           B3
P4           426           B5

--- Worst Fit ---
Process No.  Process Size  Block No.
P1           212           B5
P2           417           B2
P3           112           B5
P4           426           Not Allocated


## Practical 9: Page Replacement Algorithms: FIFO, LRU and Optimal

**Aim:** To implement and compare the FIFO, LRU and Optimal page replacement algorithms.

In [ ]:
def fifo(ref, nf):
    frames = [-1] * nf
    front = faults = 0

    for page in ref:
        if page not in frames:
            frames[front] = page
            front = (front + 1) % nf
            faults += 1
    print(f"FIFO Page Faults  = {faults}")


def lru(ref, nf):
    frames = [-1] * nf
    last_used = [0] * nf
    filled = faults = 0

    for i in range(len(ref)):
        pos = frames.index(ref[i]) if ref[i] in frames else -1

        if pos == -1:
            faults += 1
            if filled < nf:
                frames[filled] = ref[i]
                last_used[filled] = i
                filled += 1
            else:
                # find least recently used frame
                lru_idx = 0
                for j in range(1, nf):
                    if last_used[j] < last_used[lru_idx]:
                        lru_idx = j
                frames[lru_idx] = ref[i]
                last_used[lru_idx] = i
        else:
            last_used[pos] = i
    print(f"LRU Page Faults   = {faults}")


def optimal(ref, nf):
    n = len(ref)
    frames = [-1] * nf
    filled = faults = 0

    for i in range(n):
        if ref[i] in frames[:filled]:
            continue

        faults += 1
        if filled < nf:
            frames[filled] = ref[i]
            filled += 1
            continue

        # find the frame whose page is used farthest in future (or never)
        replace_idx, farthest = -1, -1
        for j in range(nf):
            future = ref[i + 1:]
            k = i + 1 + future.index(frames[j]) if frames[j] in future else n
            if k == n:                      # never used again
                replace_idx = j
                break
            if k > farthest:
                farthest = k
                replace_idx = j
        frames[replace_idx] = ref[i]
    print(f"Optimal Page Faults = {faults}")


def main():
    n = int(input("Enter number of pages in reference string: "))
    ref = list(map(int, input("Enter the reference string: ").split()))[:n]

    nf = int(input("Enter number of frames: "))

    fifo(ref, nf)
    lru(ref, nf)
    optimal(ref, nf)


if __name__ == "__main__":
    main()

Enter number of pages in reference string: 12
Enter the reference string: 7 0 1 2 0 3 0 4 2 3 0 3
Enter number of frames: 3
FIFO Page Faults  = 10
LRU Page Faults   = 9
Optimal Page Faults = 7


## Practical 10: Disk Scheduling Algorithms: FCFS, SSTF, SCAN and C-SCAN

**Aim:** To implement and compare the FCFS, SSTF, SCAN and C-SCAN disk scheduling algorithms.

In [ ]:
def fcfs(req, head):
    total, cur = 0, head
    for r in req:
        total += abs(cur - r)
        cur = r
    print(f"FCFS   total head movement = {total}")


def sstf(req, head):
    n = len(req)
    visited = [False] * n
    total, cur = 0, head
    for _ in range(n):
        idx, best = -1, 1 << 30
        for i in range(n):
            if not visited[i] and abs(cur - req[i]) < best:
                best = abs(cur - req[i])
                idx = i
        visited[idx] = True
        total += best
        cur = req[idx]
    print(f"SSTF   total head movement = {total}")


def scan(req, head, disk_size):
    n = len(req)
    arr = sorted(req + [head])
    idx = arr.index(head)

    total, cur = 0, head
    # go towards the higher end first
    for i in range(idx + 1, n + 1):
        total += abs(cur - arr[i])
        cur = arr[i]
    total += abs(cur - (disk_size - 1))
    cur = disk_size - 1
    for i in range(idx - 1, -1, -1):
        total += abs(cur - arr[i])
        cur = arr[i]

    print(f"SCAN   total head movement = {total}")


def cscan(req, head, disk_size):
    n = len(req)
    arr = sorted(req + [head])
    idx = arr.index(head)

    total, cur = 0, head
    for i in range(idx + 1, n + 1):
        total += abs(cur - arr[i])
        cur = arr[i]
    total += abs(cur - (disk_size - 1))   # go to the end
    total += disk_size - 1                # jump to 0 (cylinder 0)
    cur = 0
    for i in range(idx):
        total += abs(cur - arr[i])
        cur = arr[i]

    print(f"C-SCAN total head movement = {total}")


def main():
    n = int(input("Enter number of requests: "))
    req = list(map(int, input("Enter the requests: ").split()))[:n]
    head = int(input("Enter initial head position: "))
    disk_size = int(input("Enter disk size (max cylinder + 1): "))

    fcfs(req, head)
    sstf(req, head)
    scan(req, head, disk_size)
    cscan(req, head, disk_size)


if __name__ == "__main__":
    main()

Enter number of requests: 8
Enter the requests: 98 183 37 122 14 124 65 67
Enter initial head position: 53
Enter disk size (max cylinder + 1): 200
FCFS   total head movement = 640
SSTF   total head movement = 236
SCAN   total head movement = 331
C-SCAN total head movement = 382


## Practical 11: File Allocation Methods: Contiguous, Linked and Indexed

**Aim:** To study and demonstrate the contiguous, linked and indexed file allocation methods.

In [ ]:
MAXBLOCKS = 30

disk = [0] * MAXBLOCKS   # 0 = free, 1 = allocated


def reset_disk():
    for i in range(MAXBLOCKS):
        disk[i] = 0


def contiguous_allocation():
    reset_disk()
    print("\n--- Contiguous Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        start, length = map(int, input(f"File {f + 1} - starting block, length: ").split())

        ok = all(i < MAXBLOCKS and disk[i] == 0 for i in range(start, start + length))
        if not ok:
            print("Allocation failed: blocks not free/out of range.")
            continue

        for i in range(start, start + length):
            disk[i] = 1
        print(f"File {f + 1} allocated blocks {start} to {start + length - 1}")


def linked_allocation():
    reset_disk()
    print("\n--- Linked Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        length = int(input(f"File {f + 1} - number of blocks: "))

        blocks = []
        for i in range(MAXBLOCKS):
            if len(blocks) == length:
                break
            if disk[i] == 0:
                disk[i] = 1
                blocks.append(i)
        print(f"File {f + 1} allocated blocks:", " -> ".join(map(str, blocks)), "-> (NULL)")
        if len(blocks) < length:
            print("Warning: disk full, could not allocate all blocks.")


def indexed_allocation():
    reset_disk()
    print("\n--- Indexed Allocation ---")
    nf = int(input("Enter number of files: "))

    for f in range(nf):
        length = int(input(f"File {f + 1} - number of data blocks needed: "))

        index_block = -1
        for i in range(MAXBLOCKS):
            if disk[i] == 0:
                index_block = i
                disk[i] = 1
                break

        data_blocks = []
        for i in range(MAXBLOCKS):
            if len(data_blocks) == length:
                break
            if disk[i] == 0:
                disk[i] = 1
                data_blocks.append(i)
        print(f"File {f + 1} - Index block: {index_block}, Data blocks:", *data_blocks)


def main():
    print("Menu:\n1. Contiguous Allocation\n2. Linked Allocation")
    print("3. Indexed Allocation")
    choice = int(input("Enter choice: "))

    if choice == 1:
        contiguous_allocation()
    elif choice == 2:
        linked_allocation()
    elif choice == 3:
        indexed_allocation()
    else:
        print("Invalid choice")


if __name__ == "__main__":
    main()

Menu:
1. Contiguous Allocation
2. Linked Allocation
3. Indexed Allocation
Enter choice: 3

--- Indexed Allocation ---
Enter number of files: 2
File 1 - number of data blocks needed: 4
File 1 - Index block: 0, Data blocks: 1 2 3 4
File 2 - number of data blocks needed: 3
File 2 - Index block: 5, Data blocks: 6 7 8
